<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item's state, as observed at a single decision date.** The date's range is from 2025-01-27 to 2026-06-30

**Tables**: I will be using tables dim_content and fact_content_daily_performance

**Time window: a rolling 60-day lookback, not a fixed calendar month.** Anchored at an explicit
as-of date, **2026-05-31**, a rather fresh panel.
I split the 60 days before it into two 30-day windows, the same last-30-vs-prev-30 momentum shape
the starter CSV's own `trend_direction` uses.

- `prev_30` = `[2026-04-01, 2026-05-01)`
- `last_30` = `[2026-05-01, 2026-05-31)`

**Decision moment: 2026-05-01**: the `prev_30`/`last_30` boundary. Anything a feature uses must be
knowable by then: a static `dim_content` attribute, or a `prev_30` aggregate. `last_30` exists only
to help define the label; using it as a feature is leakage by construction (section 3e).

Three guards before trusting a content item's window pair at all (both computed from
`fact_content_daily_performance` alone — the panel is unbalanced across clients, so this matters):
**(a)** it needs continuous history back to `2026-04-01` (the start of `prev_30`), content that only
started reporting partway through gets dropped, not zero-filled; **(b)** `prev_30` impressions must
be `>= 100` before a percent-change off that baseline means anything (for instance, a 3→6 growth is a meaningless "100% increase."); **(c)** The content should not be updated after the decision moment.

Separate from all of the above: the single-month partition (`month=2026-03`) used in section 3a-3c
is only a spot-check of three raw-table facts (grain, count, availability) it is not this
window's definition



In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

# Retrieve token securely from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT   = f"read_parquet(\'{WAREHOUSE}/dim_content.parquet\')"
FACT_MARCH    = f"read_parquet(\'{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet\')"
FACT_APRIL_MAY  = (
    f"read_parquet(['{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet', "
    f"'{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet', "
    f"'{WAREHOUSE}/fact_content_daily_performance/month=2026-05/*.parquet'])"
)

print("DuckDB ready. Tables pointed at:")
print(" dim_content            ->", DIM_CONTENT)
print(" fact, March only       -> 3a-3c spot-check ->", FACT_MARCH)
print(" fact, April+May        -> 3d-3e window build ->", FACT_APRIL_MAY)



DuckDB ready. Tables pointed at:
 dim_content            -> read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
 fact, March only       -> 3a-3c spot-check -> read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
 fact, April+May        -> 3d-3e window build -> read_parquet(['hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet', 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet', 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet'])


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context (join/group/filter only, never a feature):** `client_hash_id`, `content_hash_id` (both
tables), plus `keyword_hash_id` / `url_hash_id` (`dim_content`): pseudonyms, grouping keys only.

**Label / proxy (never a feature):** `is_declining`: 1 when `last_30_impressions` falls more than
20% below `prev_30_impressions`, else 0 (mirrors the starter CSV's own `trend_direction` rule). `last_30_impressions` and the resulting
`impressions_pct_change` are label-derived and must never be features (section 3e shows exactly why).


**Features (5, all knowable by the 2026-06-01 decision moment):** `content_age_days_at_decision`,
`days_since_last_update_at_decision` (select only rows with > 0), `word_count` (all from `dim_content`, static, set long before
May even starts), plus `prev_30_avg_position` and `prev_30_impressions` (aggregated over
`prev_30` = 2026-05-02→06-01, from `fact_content_daily_performance`). One "available when?" line
per feature is in section 3d.

**Excluded (not label, not a feature — with a why):** `provider_used` / `model_used` from
`dim_content` which AI provider/model produced the content. That's pipeline metadata about *how*
the content was made, not a search or reader signal

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
field_buckets = {
    "context":  ["client_hash_id", "content_hash_id", "keyword_hash_id", "url_hash_id"],
    "label (never a feature)": ["is_declining (derived)", "last_30_impressions (derived)",
                                 "impressions_pct_change (derived)"],
    "features (5)": ["content_age_days_at_decision", "days_since_last_update_at_decision",
                      "word_count", "prev_30_avg_position", "prev_30_impressions"],
    "excluded": ["provider_used", "model_used"],
}
for bucket, cols in field_buckets.items():
    print(f"{bucket:28s}: {cols}")

context                     : ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id']
label (never a feature)     : ['is_declining (derived)', 'last_30_impressions (derived)', 'impressions_pct_change (derived)']
features (5)                : ['content_age_days_at_decision', 'days_since_last_update_at_decision', 'word_count', 'prev_30_avg_position', 'prev_30_impressions']
excluded                    : ['provider_used', 'model_used']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries (3a-3c) on the single `month=2026-03` spot-check partition (`FACT_MARCH`),
then the five-feature frame and the deliberate leak (3d-3e) on the real rolling window
(`FACT_MAY_JUNE`).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 3a. Grain — one row really is report_date × content

In [14]:
grain_check = con.sql(f"""
    SELECT report_date, content_hash_id, COUNT(*) AS c
    FROM {FACT_MARCH}
    GROUP BY report_date, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Duplicate (report_date, content_hash_id) combinations found: {len(grain_check)}")
print("-> 0 means the documented grain holds for this slice.")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (report_date, content_hash_id) combinations found: 0
-> 0 means the documented grain holds for this slice.


,report_date,content_hash_id,c


### 3b. Row count and date span for this slice

In [15]:
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_MARCH}
""").df()
counts

print("In this slice, we have 9841378 rows for 331437	different contents.")
print("The time for this slice spans the March of 2026")



In this slice, we have 9841378 rows for 331437	different contents.
The time for this slice spans the March of 2026


### 3c. Availability — filter with IS TRUE, count survivors

We need both ga4 and gsc data to be available

In [16]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)  AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END)  AS ga4_flag_null_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)  AS gsc_available_rows,
        SUM(CASE WHEN gsc_data_available IS NULL THEN 1 ELSE 0 END)  AS gsc_flag_null_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_available_rows
    FROM {FACT_MARCH}
""").df()

availability["ga4_available_pct"] = (availability["ga4_available_rows"] / availability["total_rows"] * 100).round(1)
availability["gsc_available_pct"] = (availability["gsc_available_rows"] / availability["total_rows"] * 100).round(1)
availability["both_available_pct"] = (availability["both_available_rows"] / availability["total_rows"] * 100).round(1)
availability

print("Only 364347 rows (3.7%) survived")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Only 364347 rows (3.7%) survived


### 3d. Five features — one "available at decision moment" line each

Built on `FACT_FEB_MAR` (the real rolling window), with both guards from section 1 applied:
history back to 2026-04-01, and `prev_30_impressions >= 100`.

- `content_age_days_at_decision` — days between `content_created_date` and 2026-05-01. Available:
  a past, immutable event, recorded long before the decision moment.
- `days_since_last_update_at_decision` — days between `content_updated_date` and 2026-05-01.
  Available: also a past event, because we will select only the rows with a content that were last updated before the decision moment.
- `word_count` — a static content property from `dim_content`. Available: set when the content was
  authored/last edited, not something search behavior changes after the fact.
- `prev_30_avg_position` — mean `gsc_avg_position` over `prev_30` (2026-04-01→05-01). Available:
  entirely `prev_30` data, which ends exactly at the decision moment.
- `prev_30_impressions` — sum of `gsc_impressions` over the same `prev_30` window. Available: same
  reasoning, and the same guard-(b) column.

In [17]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT DATE \'2026-05-31\' AS as_of_date
    ),
    per_item AS (
        SELECT f.client_hash_id, f.content_hash_id,
               MIN(f.report_date) AS first_seen,
               SUM(CASE WHEN f.report_date >= b.as_of_date - INTERVAL 30 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS last_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS prev_30_impressions,
               AVG(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_avg_position END)       AS prev_30_avg_position
        FROM {FACT_APRIL_MAY} f, bounds b
        GROUP BY 1, 2
    )
    SELECT p.*, d.content_created_date, d.content_updated_date, d.word_count
    FROM per_item p
    JOIN {DIM_CONTENT} d USING (content_hash_id)
    WHERE p.first_seen <= DATE \'2026-05-31\' - INTERVAL 60 DAY   -- guard (a): full prev_30 history
      AND p.prev_30_impressions >= 100                             -- guard (b): activity floor
""").df()

decision_moment = pd.Timestamp("2026-05-01")
features["content_created_date"] = pd.to_datetime(features["content_created_date"])
features["content_updated_date"] = pd.to_datetime(features["content_updated_date"])
features["content_age_days_at_decision"] = (decision_moment - features["content_created_date"]).dt.days
features["days_since_last_update_at_decision"] = (decision_moment - features["content_updated_date"]).dt.days

features = features[features["days_since_last_update_at_decision"] > 0] # the guard (c)

print(f"Feature frame: {len(features):,} content items surviving all three guards.")

features[["content_hash_id", "content_age_days_at_decision", "days_since_last_update_at_decision",
          "word_count", "prev_30_avg_position", "prev_30_impressions",
          "last_30_impressions"]].head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 18,918 content items surviving all three guards.


,content_hash_id,content_age_days_at_decision,days_since_last_update_at_decision,word_count,prev_30_avg_position,prev_30_impressions,last_30_impressions
1843,content_8ef4a23c8dfdcbf8,323,65,<NA>,25.331508,115.0,178.0
1844,content_d3c09107f60a3cba,323,65,<NA>,17.854615,501.0,558.0
1845,content_7c3e8f9e825838cf,323,65,<NA>,18.458797,549.0,564.0
1846,content_441806c0f1d0fb13,323,65,<NA>,2.227627,105.0,30.0
1847,content_61e03ed11bd944f8,323,65,<NA>,13.677377,1037.0,1525.0


### 3e. The trap — add a label-derived column, watch the score jump, then delete it

Adding `impressions_pct_change`, the exact last-30-vs-prev-30 percent change that DEFINES
`is_declining`, as if it were a sixth feature. It shouldn't survive in the final feature set; the
point is to watch what a classifier does while it's still there.

In [18]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

work = features.copy()
work = work.dropna()
print(f"Number of rows without missing values {len(work)}")
work["impressions_pct_change"] = (
    (work["last_30_impressions"] - work["prev_30_impressions"]) / work["prev_30_impressions"]
)
work["is_declining"] = (work["impressions_pct_change"] < -0.20).astype(int)

honest_features = ["content_age_days_at_decision", "days_since_last_update_at_decision",
                    "word_count", "prev_30_avg_position", "prev_30_impressions"]
leak_feature = "impressions_pct_change"

X_honest = work[honest_features]
X_leaked = work[honest_features + [leak_feature]]

y = work["is_declining"]
groups = work["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X_honest, y, groups))

def client_grouped_auc(X):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]

    return roc_auc_score(yte, proba)

honest_auc = client_grouped_auc(X_honest)
leaked_auc = client_grouped_auc(X_leaked)

print(f"Honest 5-feature AUC:                {honest_auc:.3f}")
print(f"With impressions_pct_change added:   {leaked_auc:.3f}   <- jumps sharply to 1.0")
print()
print("Deleting impressions_pct_change now. Keeping the honest number:", round(honest_auc, 3))

Number of rows without missing values 9447
Honest 5-feature AUC:                0.477
With impressions_pct_change added:   1.000   <- jumps sharply to 1.0

Deleting impressions_pct_change now. Keeping the honest number: 0.477


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



**The three guards in section 1 impose some limitations**: Anything without continuous history back to 2026-04-01 (new content, or
a client whose access started partway through) or with fewer than 100 `prev_30` impressions gets
dropped outright, not zero-filled. The contents which got updated in May or June are not taken into consideration.

Therefore the results cannot be generalized to the contents that have little impressions, or got updated recently.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.